In [1]:
import pandas as pd
import numpy as np

import mosmapapi

In [2]:
# !pip install openpyxl

In [3]:
df = pd.read_excel("../data/moscow_transformed.xlsx").set_index(['ID на сайте', 'Источник'], drop=False).sort_index()
df

,,ID на сайте,Источник,Название,Цена,Дата,Тип автора,Метро/Район,Адрес,lat,lng,URL,Ссылки на картинки,"Расстояние до метро, км",Этаж,Этажность здания,Вид объекта,Общая площадь
ID на сайте,Источник,,,,,,,,,,,,,,,,,
295681856,cian.ru,295681856,cian.ru,"Торговая площадь в Москва Трифоновская ул., 12...",1499904,2025-08-23 22:24:58,Агентство,Марьина Роща,"Трифоновская ул., 12",55.788196,37.615066,https://www.cian.ru/rent/commercial/295681856,['https://images.cdn-cian.ru/images/2286035094...,0.833,-2.0,2.0,Торговое / Свободного назначения,1208.0
303266548,cian.ru,303266548,cian.ru,Помещение свободного назначения в Москва Голов...,325000,2025-08-16 00:22:09,Агентство,Водный Стадион,"Головинское ш., 4",55.839021,37.489500,https://www.cian.ru/rent/commercial/303266548,['https://images.cdn-cian.ru/images/2191018071...,0.250,1.0,5.0,Торговое / Свободного назначения,38.0
303765608,cian.ru,303765608,cian.ru,"Здание в Москва Средний Овчинниковский пер., 8...",959400,2025-08-16 00:21:59,Агентство,Новокузнецкая,"Средний Овчинниковский пер., 8С2",55.744860,37.630014,https://www.cian.ru/rent/commercial/303765608,['https://images.cdn-cian.ru/images/2201493152...,0.417,NaN,3.0,Здание,246.0
304629141,cian.ru,304629141,cian.ru,Помещение свободного назначения в Москва Енисе...,60000,2025-08-16 00:22:16,Агентство,Свиблово,"Енисейская ул., 5",55.858584,37.659847,https://www.cian.ru/rent/commercial/304629141,['https://images.cdn-cian.ru/images/2217919212...,0.583,1.0,21.0,Торговое / Свободного назначения,25.0
304669428,cian.ru,304669428,cian.ru,Помещение свободного назначения в Москва Светл...,955200,2025-08-12 00:21:53,Агентство,Стрешнево,"Светлый проезд, 2",55.811680,37.489455,https://www.cian.ru/rent/commercial/304669428,['https://images.cdn-cian.ru/images/nezhiloe-p...,0.167,2.0,3.0,Торговое / Свободного назначения,597.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9028468365467385856,realty.yandex.ru,9028468365467385856,realty.yandex.ru,Офис (74 м²),285000,2025-12-04 13:59:03,Агентство,Нагатинская,"Каширское шоссе, 3к2с12",55.673424,37.631855,https://realty.ya.ru/offer/9028468365467386113/,['https://avatars.mds.yandex.net/get-realty-of...,1.583,5.0,6.0,Офисное помещение,74.0
9033994393054442496,realty.yandex.ru,9033994393054442496,realty.yandex.ru,Офис (32 м²),140000,2025-10-29 13:59:12,Агентство,Генерала Тюленева,"улица Академика Варги, 8к1",55.630142,37.474453,https://realty.ya.ru/offer/9033994393054442241/,['https://avatars.mds.yandex.net/get-realty-of...,1.000,9.0,15.0,Офисное помещение,32.0
9051973490662697984,realty.yandex.ru,9051973490662697984,realty.yandex.ru,Офис (70 м²),175250,2025-11-11 10:59:12,Агентство,Шаболовская,"2-й Верхний Михайловский проезд, 9с2",55.708508,37.602590,https://realty.ya.ru/offer/9051973490662697984/,['https://avatars.mds.yandex.net/get-realty-of...,1.333,3.0,6.0,Офисное помещение,70.0


In [4]:
big_index = list(df.index)
len(big_index)

9070

In [5]:
new_df = pd.read_csv("../data/moscow_super_transformed.csv")
if (len(new_df) < 10):
    new_df = pd.DataFrame()
else:
    new_df = new_df.set_index(['ID на сайте', 'Источник'])
existing_indices = [set(new_df.index), set(new_df.index)]

def get_batch_data(l: int, r: int, radius: int, id: int): # [l, r)
    global new_df
    ls = []
    for i in range(l, r):
        val = df.loc[big_index[i]]
        if big_index[i] in existing_indices[id]:
            continue
        lat = val['lat'].iloc[0]
        lng = val['lng'].iloc[0]
        radius_new_data = mosmapapi.get_data_radius(lat, lng, radius)
        _ = pd.DataFrame(radius_new_data, index=[0])
        _['ID на сайте'] = val['ID на сайте'].iloc[0]
        _['Источник'] = val['Источник'].iloc[0]
        existing_indices[id].add(big_index[i])
        ls.append(_)
    if len(ls) == 0:
        return None
    return pd.concat(ls).set_index(['ID на сайте', 'Источник'])

def get_all_data(batch_size: int, l: int, r: int):
    global new_df
    for i in range(l, r, batch_size):
        dt1 = get_batch_data(i, min(i + batch_size, r), 300, 0)
        dt2 = get_batch_data(i, min(i + batch_size, r), 600, 1)
        if dt1 is None or dt2 is None:
            continue
        _ = pd.concat([dt1, dt2.drop(['district_price', 'district_name'], axis=1)], axis=1)
        new_df = pd.concat([new_df, _], axis=0)
        new_df.to_csv("../data/moscow_super_transformed.csv")

In [6]:
df.loc[big_index[0]]['lat'].iloc[0]

np.float64(55.788196)

In [7]:
get_all_data(10, 0, 9000)

C:\Users\misha\AppData\Local\Temp\ipykernel_12468\884549177.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(ls).set_index(['ID на сайте', 'Источник'])
C:\Users\misha\AppData\Local\Temp\ipykernel_12468\884549177.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(ls).set_index(['ID на сайте', 'Источник'])
C:\Users\misha\AppData\Local\Temp\ipykernel_12468\884549177.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is 

JSONDecodeError: Expecting value: line 1 column 1 (char 0)